In [1]:
import re
import warnings
import pyreadr
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
import statsmodels.tools.sm_exceptions as sm_exceptions
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [47]:
# loading phenotype data
phenoPath = "C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/00databases/00phenotype/00vabb/01PhenoMeta_assoc_08032025.csv"
phenoData = pd.read_csv(phenoPath)
# Create suffixes
suffixes = ["_9", "_24", "_25", "_11"]
# Replicate and modify pheno_data
phenoData = pd.concat([
    phenoData.assign(SampleID="Sample" + phenoData["SampleID"].astype(str) + suffix)
    for suffix in suffixes
], ignore_index=True)
phenoData.shape

(740, 29)

In [52]:
# Load data
dpclo = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/00deepclock-08032025.csv")
# Rename and keep only SampleID and KPANN_brain columns
dpclo = dpclo.rename(columns={'ID': 'SampleID', 'Predicted': 'KPANN_brain_clock'})[['SampleID', 'KPANN_brain_clock']]
# Load cell types and SVAs
cells = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/05celltype-08042025/00vabb_cellprop_svas-08042025.csv")
cells = cells.rename(columns={'name': 'SampleID'})
# Load RNAgecalc
rnaage = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/02trans_age-07272025/01vabb_rnaagecalc-07272025.csv")
rnaage = rnaage.rename(columns={'sample': 'SampleID'})
# Replace the letter between numbers with an underscore
rnaage["SampleID"] = rnaage["SampleID"].str.replace(r"(?<=\d)[A-Z]+(?=\d)", "_", regex=True)
rnaage.columns = [
    col if col == "SampleID" else f"{col}_clock"
    for col in rnaage.columns
]

In [53]:
# Merge in sequence after conversion
merged_df_com = pd.merge(phenoData, dpclo, on='SampleID')
merged_df_com = pd.merge(merged_df_com, cells, on='SampleID')
merged_df_com = pd.merge(merged_df_com, rnaage, on='SampleID')
#data_assoc = merged_df_com.dropna().reset_index(drop=True)
data_assoc = merged_df_com.drop_duplicates(subset=['SampleID'])
data_assoc.shape

(526, 254)

In [54]:
# Standardizing all covariates
# Select only numeric columns
numeric_cols = data_assoc.select_dtypes(include='number').columns
# Scale
scaler = StandardScaler()
data_assoc = data_assoc.copy()
data_assoc.loc[:, numeric_cols] = scaler.fit_transform(data_assoc[numeric_cols])

In [56]:
#------------------------------------
# Define clocks and phenotype columns
#------------------------------------
# Copy your data
binary_df = data_assoc.copy()
# Adding another age column
binary_df['Actual_Age'] = binary_df['AgeDeath']

# Define clocks and phenotype columns
brain_clocks = [col for col in binary_df.columns 
                if "brain" in col.lower() and pd.api.types.is_numeric_dtype(binary_df[col])]
brain_clocks = brain_clocks + ['Actual_Age']

# Define phenotypes
phenotype_cols = [
    "COD_drugs_bi",
    "COD_Suicide_bi",
    "PsychiatriDisor",
    "MDD",
    "PTSD",
    "LifetimeAntipsych",
    "LifetimeAnticonvulsant",
    "LifetimeAntidepress",
    "LifetimeLithium",
    "Tobbaco_ATOD",
    "Alcohol",
    "Opioid",
    "Opioid_Tox_bi",
    "Sedative_hypnotics",
    "Cannabis",
    "Amphetamine",
    "Cocaine",
    "Hallucinogens",
    "Inhalant",
    "Polysubstance"
]

# Convert Gender (and any other categorical covariates) to numeric BEFORE correlation
covariates = ['AgeDeath', 'Sex', 'RIN', 'PMI', 'ast', 'end', 'mic', 'neu', 'oli', 'opc',' W_1', 'W_2']

# Columns to convert
cols_to_convert = phenotype_cols + ['Sex']

for col in cols_to_convert:
    if col in binary_df.columns:
        # If bool, convert directly to int
        if binary_df[col].dtype == 'bool':
            binary_df[col] = binary_df[col].astype(int)
        # If object/string, convert 'Yes'/'No' and also 'Male'/'Female' for Gender
        elif binary_df[col].dtype == 'object':
            if col == 'Sex':
                binary_df[col] = binary_df[col].map({'Male': 1, 'Female': 0})
            else:
                binary_df[col] = binary_df[col].map({'Yes': 1, 'No': 0})
        else:
            # If already numeric, no change
            pass

# Quick check of unique values after conversion
for col in cols_to_convert:
    if col in binary_df.columns:
        print(f"{col} unique values: {binary_df[col].unique()}")

## Running models
# Adding a dataframe to store the results
results = []
# Running models
for clock in brain_clocks:
    for pheno in phenotype_cols:
        try:
            # Formula: clock is outcome, phenotype is predictor
            formula = f"{clock} ~ {pheno} + {' + '.join(covariates)}"
            
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                warnings.filterwarnings("ignore", category=RuntimeWarning)

                model = smf.ols(formula=formula, data=binary_df).fit()

            coef = model.params[pheno]
            pval = model.pvalues[pheno]
            conf_int = model.conf_int().loc[pheno]
            tval = model.tvalues[pheno]

            results.append({
                'Clock': clock,
                'Phenotype': pheno,
                'Beta': coef,
                'SE': model.bse[pheno],
                't-value': tval,
                'P-value': pval,
                'CI_lower_95': conf_int[0],
                'CI_upper_95': conf_int[1],
                'R2': model.rsquared,
                'Adj_R2': model.rsquared_adj,
                'N': int(model.nobs)
            })
        
        except Exception as e:
            print(f"Model failed for {clock} ~ {pheno}: {e}")
            results.append({
                'Clock': clock,
                'Phenotype': pheno,
                'Beta': np.nan,
                'SE': np.nan,
                't-value': np.nan,
                'P-value': np.nan,
                'CI_lower_95': np.nan,
                'CI_upper_95': np.nan,
                'R2': np.nan,
                'Adj_R2': np.nan,
                'N': np.nan
            })

results_df = pd.DataFrame(results)

COD_drugs_bi unique values: [ 0.  1. nan]
COD_Suicide_bi unique values: [ 0.  1. nan]
PsychiatriDisor unique values: [0 1]
MDD unique values: [0 1]
PTSD unique values: [0 1]
LifetimeAntipsych unique values: [ 0. nan  1.]
LifetimeAnticonvulsant unique values: [ 0. nan  1.]
LifetimeAntidepress unique values: [ 0.  1. nan]
LifetimeLithium unique values: [ 0. nan  1.]
Tobbaco_ATOD unique values: [ 0.  1. nan]
Alcohol unique values: [0 1]
Opioid unique values: [0 1]
Opioid_Tox_bi unique values: [ 0.  1. nan]
Sedative_hypnotics unique values: [0 1]
Cannabis unique values: [0 1]
Amphetamine unique values: [0 1]
Cocaine unique values: [0 1]
Hallucinogens unique values: [0 1]
Inhalant unique values: [0]
Polysubstance unique values: [0 1]
Sex unique values: [1 0]


In [57]:
# Save results with tissue information
results_df.to_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/03vabb_assoc-09012025.csv", index=False)
results_df

,Clock,Phenotype,Beta,SE,t-value,P-value,CI_lower_95,CI_upper_95,R2,Adj_R2,N
0,KPANN_brain_clock,COD_drugs_bi,-1.821515e-01,9.504127e-02,-1.916552e+00,5.585562e-02,-3.688739e-01,4.570785e-03,0.088929,0.065614,522
1,KPANN_brain_clock,COD_Suicide_bi,-3.275224e-01,1.349827e-01,-2.426402e+00,1.559635e-02,-5.927155e-01,-6.232932e-02,0.092854,0.069640,522
2,KPANN_brain_clock,PsychiatriDisor,-4.452647e-02,9.414461e-02,-4.729582e-01,6.364446e-01,-2.294837e-01,1.404308e-01,0.081069,0.057736,526
3,KPANN_brain_clock,MDD,-2.590634e-02,8.832283e-02,-2.933142e-01,7.694008e-01,-1.994261e-01,1.476134e-01,0.080822,0.057483,526
4,KPANN_brain_clock,PTSD,-9.816951e-02,9.117026e-02,-1.076771e+00,2.820896e-01,-2.772833e-01,8.094432e-02,0.082744,0.059455,526
...,...,...,...,...,...,...,...,...,...,...,...
195,Actual_Age,Amphetamine,-2.684051e-15,5.415436e-16,-4.956297e+00,9.782551e-07,-3.747972e-15,-1.620130e-15,1.000000,1.000000,526
196,Actual_Age,Cocaine,2.480655e-16,1.276306e-16,1.943620e+00,5.248881e-02,-2.678692e-18,4.988096e-16,1.000000,1.000000,526
197,Actual_Age,Hallucinogens,-2.684051e-15,5.415436e-16,-4.956297e+00,9.782551e-07,-3.747972e-15,-1.620130e-15,1.000000,1.000000,526
198,Actual_Age,Inhalant,-1.909031e-16,5.079823e-32,-3.758066e+15,0.000000e+00,-1.909031e-16,-1.909031e-16,1.000000,1.000000,526


In [58]:
# -------------------------------
# Define a function for stepwise regression for each brain clock
# -------------------------------
def forward_stepwise_selection(data, response, predictors, covariates=[], 
                               threshold_in=0.05, verbose=False):
    """
    Forward stepwise regression: add predictors if p-value < threshold_in.
    Covariates are always included.
    """
    included = list(covariates)  # start with covariates
    # Remove covariates from predictors list to avoid adding again
    predictors = [p for p in predictors if p not in covariates]

    while True:
        changed = False
        excluded = [p for p in predictors if p not in included]
        new_pvals = pd.Series(dtype=float)

        for new_col in excluded:
            try:
                formula = "{} ~ {}".format(response, " + ".join(included + [new_col]))
                model = smf.ols(formula=formula, data=data).fit()
                if new_col in model.pvalues:
                    new_pvals.loc[new_col] = model.pvalues[new_col]
            except Exception:
                continue

        if not new_pvals.empty:
            best_pval = new_pvals.min()
            if best_pval < threshold_in:
                best_feature = new_pvals.idxmin()
                included.append(best_feature)
                changed = True
                if verbose:
                    print(f"Add {best_feature} with p={best_pval:.6f}")

        if not changed:
            break

    return included

In [59]:
# Strip names in phenotype_cols and covariates
binary_df.columns = binary_df.columns.str.strip()
phenotype_cols = [c.strip() for c in phenotype_cols]
covariates = [c.strip() for c in covariates]

# Run forward stepwise for each brain clock
# -------------------------------
results_step = []
for clock in brain_clocks:
    try:
        selected = forward_stepwise_selection(
            data=binary_df,
            response=clock,
            predictors=phenotype_cols,
            covariates=covariates,
            verbose=False
        )

        if not selected:  # safety check
            print(f"No predictors selected for {clock}")
            continue

        formula = f"{clock} ~ {' + '.join(selected)}"
        model = smf.ols(formula=formula, data=binary_df).fit()

        for var in selected:
            if var == "Intercept":
                continue
            try:
                coef = model.params[var]
                pval = model.pvalues[var]
                conf_int = model.conf_int().loc[var]
                tval = model.tvalues[var]

                results_step.append({
                    'Clock': clock,
                    'Variable': var,
                    'Beta': coef,
                    'SE': model.bse[var],
                    't-value': tval,
                    'P-value': pval,
                    'CI_lower_95': conf_int[0],
                    'CI_upper_95': conf_int[1],
                    'R2': model.rsquared,
                    'Adj_R2': model.rsquared_adj,
                    'N': int(model.nobs),
                    'Selected_Model': formula
                })
            except KeyError:
                print(f"⚠️ Variable {var} not found in model for {clock}")

    except Exception as e:
        print(f"Stepwise failed for {clock}: {e}")
        results_step.append({
            'Clock': clock,
            'Variable': None,
            'Beta': np.nan,
            'SE': np.nan,
            't-value': np.nan,
            'P-value': np.nan,
            'CI_lower_95': np.nan,
            'CI_upper_95': np.nan,
            'R2': np.nan,
            'Adj_R2': np.nan,
            'N': np.nan,
            'Selected_Model': None
        })

# -------------------------------
# Save results
# -------------------------------
results_df_step = pd.DataFrame(results_step)

In [60]:
# Save results with tissue information
results_df_step.to_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/03vabb_assoc_forward-09012025.csv", index=False)
results_df_step

,Clock,Variable,Beta,SE,t-value,P-value,CI_lower_95,CI_upper_95,R2,Adj_R2,N,Selected_Model
0,KPANN_brain_clock,AgeDeath,2.525696e-01,4.330327e-02,5.832576e+00,9.721861e-09,1.674941e-01,3.376451e-01,0.092854,0.06964,522,KPANN_brain_clock ~ AgeDeath + Sex + RIN + PMI...
1,KPANN_brain_clock,Sex,2.735074e-02,9.190697e-02,2.975915e-01,7.661366e-01,-1.532138e-01,2.079153e-01,0.092854,0.06964,522,KPANN_brain_clock ~ AgeDeath + Sex + RIN + PMI...
2,KPANN_brain_clock,RIN,-5.174864e-03,4.469634e-02,-1.157783e-01,9.078740e-01,-9.298729e-02,8.263756e-02,0.092854,0.06964,522,KPANN_brain_clock ~ AgeDeath + Sex + RIN + PMI...
3,KPANN_brain_clock,PMI,-3.148870e-02,4.366403e-02,-7.211590e-01,4.711435e-01,-1.172730e-01,5.429560e-02,0.092854,0.06964,522,KPANN_brain_clock ~ AgeDeath + Sex + RIN + PMI...
4,KPANN_brain_clock,ast,-3.880905e-02,9.048666e-02,-4.288925e-01,6.681833e-01,-2.165832e-01,1.389651e-01,0.092854,0.06964,522,KPANN_brain_clock ~ AgeDeath + Sex + RIN + PMI...
...,...,...,...,...,...,...,...,...,...,...,...,...
160,Actual_Age,Amphetamine,5.756888e-18,3.287101e-32,1.751357e+14,0.000000e+00,5.756888e-18,5.756888e-18,1.000000,1.00000,273,Actual_Age ~ AgeDeath + Sex + RIN + PMI + ast ...
161,Actual_Age,MDD,-4.068775e-16,2.406542e-16,-1.690714e+00,9.211323e-02,-8.808003e-16,6.704533e-17,1.000000,1.00000,273,Actual_Age ~ AgeDeath + Sex + RIN + PMI + ast ...
162,Actual_Age,Polysubstance,-4.410779e-17,1.538076e-16,-2.867725e-01,7.745194e-01,-3.470026e-16,2.587871e-16,1.000000,1.00000,273,Actual_Age ~ AgeDeath + Sex + RIN + PMI + ast ...
163,Actual_Age,PTSD,-9.464082e-16,1.619898e-16,-5.842395e+00,1.563834e-08,-1.265416e-15,-6.274000e-16,1.000000,1.00000,273,Actual_Age ~ AgeDeath + Sex + RIN + PMI + ast ...


In [61]:
# Strip names in phenotype_cols and covariates
# Convert Gender (and any other categorical covariates) to numeric BEFORE correlation
covariates = ['AgeDeath', 'Age2', 'Sex', 'RIN', 'PMI', 'ast', 'end', 'mic', 'neu', 'oli', 'opc',' W_1', 'W_2']
# Add the Age2 variable
binary_df['Age2'] = binary_df['AgeDeath']**2
binary_df.columns = binary_df.columns.str.strip()
phenotype_cols = [c.strip() for c in phenotype_cols]
covariates = [c.strip() for c in covariates]

# Run forward stepwise for each brain clock
# -------------------------------
results_step_age2 = []
for clock in brain_clocks:
    try:
        selected = forward_stepwise_selection(
            data=binary_df,
            response=clock,
            predictors=phenotype_cols,
            covariates=covariates,
            verbose=False
        )

        if not selected:  # safety check
            print(f"No predictors selected for {clock}")
            continue

        formula = f"{clock} ~ {' + '.join(selected)}"
        model = smf.ols(formula=formula, data=binary_df).fit()

        for var in selected:
            if var == "Intercept":
                continue
            try:
                coef = model.params[var]
                pval = model.pvalues[var]
                conf_int = model.conf_int().loc[var]
                tval = model.tvalues[var]

                results_step_age2.append({
                    'Clock': clock,
                    'Variable': var,
                    'Beta': coef,
                    'SE': model.bse[var],
                    't-value': tval,
                    'P-value': pval,
                    'CI_lower_95': conf_int[0],
                    'CI_upper_95': conf_int[1],
                    'R2': model.rsquared,
                    'Adj_R2': model.rsquared_adj,
                    'N': int(model.nobs),
                    'Selected_Model': formula
                })
            except KeyError:
                print(f"⚠️ Variable {var} not found in model for {clock}")

    except Exception as e:
        print(f"Stepwise failed for {clock}: {e}")
        results_step_age2.append({
            'Clock': clock,
            'Variable': None,
            'Beta': np.nan,
            'SE': np.nan,
            't-value': np.nan,
            'P-value': np.nan,
            'CI_lower_95': np.nan,
            'CI_upper_95': np.nan,
            'R2': np.nan,
            'Adj_R2': np.nan,
            'N': np.nan,
            'Selected_Model': None
        })

# -------------------------------
# Save results
# -------------------------------
results_df_step_age2 = pd.DataFrame(results_step_age2)

In [62]:
# Save results with tissue information
results_df_step_age2.to_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/03vabb_assoc_forward_age2-09012025.csv", index=False)
results_df_step_age2

,Clock,Variable,Beta,SE,t-value,P-value,CI_lower_95,CI_upper_95,R2,Adj_R2,N,Selected_Model
0,KPANN_brain_clock,AgeDeath,2.496118e-01,4.348982e-02,5.739547,1.635675e-08,1.641694e-01,3.350543e-01,0.093918,0.068898,522,KPANN_brain_clock ~ AgeDeath + Age2 + Sex + RI...
1,KPANN_brain_clock,Age2,-2.444585e-02,3.168427e-02,-0.771545,4.407432e-01,-8.669447e-02,3.780277e-02,0.093918,0.068898,522,KPANN_brain_clock ~ AgeDeath + Age2 + Sex + RI...
2,KPANN_brain_clock,Sex,2.981793e-02,9.199919e-02,0.324111,7.459878e-01,-1.509287e-01,2.105645e-01,0.093918,0.068898,522,KPANN_brain_clock ~ AgeDeath + Age2 + Sex + RI...
3,KPANN_brain_clock,RIN,-4.969851e-03,4.471494e-02,-0.111145,9.115452e-01,-9.281924e-02,8.287954e-02,0.093918,0.068898,522,KPANN_brain_clock ~ AgeDeath + Age2 + Sex + RI...
4,KPANN_brain_clock,PMI,-3.051223e-02,4.369976e-02,-0.698224,4.853571e-01,-1.163671e-01,5.534268e-02,0.093918,0.068898,522,KPANN_brain_clock ~ AgeDeath + Age2 + Sex + RI...
...,...,...,...,...,...,...,...,...,...,...,...,...
166,Actual_Age,LifetimeAntidepress,-2.954471e-16,1.335939e-16,-2.211532,2.784943e-02,-5.584831e-16,-3.241111e-17,1.000000,1.000000,285,Actual_Age ~ AgeDeath + Age2 + Sex + RIN + PMI...
167,Actual_Age,Opioid,4.095284e-17,1.386727e-16,0.295320,7.679797e-01,-2.320829e-16,3.139886e-16,1.000000,1.000000,285,Actual_Age ~ AgeDeath + Age2 + Sex + RIN + PMI...
168,Actual_Age,COD_drugs_bi,-2.299726e-16,1.694936e-16,-1.356822,1.759882e-01,-5.636924e-16,1.037472e-16,1.000000,1.000000,285,Actual_Age ~ AgeDeath + Age2 + Sex + RIN + PMI...
169,Actual_Age,Polysubstance,1.601594e-16,1.574675e-16,1.017095,3.100325e-01,-1.498819e-16,4.702006e-16,1.000000,1.000000,285,Actual_Age ~ AgeDeath + Age2 + Sex + RIN + PMI...


In [114]:
# -------------------------------
# Extract brain region from SampleID
# -------------------------------
binary_df2 = data_assoc.copy()
# Define clocks and phenotype columns
brain_clocks = [col for col in binary_df.columns 
                if "brain" in col.lower() and pd.api.types.is_numeric_dtype(binary_df[col])]
brain_clocks = brain_clocks
# Define phenotypes
phenotype_cols = [
    "COD_drugs_bi",
    "COD_Suicide_bi",
    "PsychiatriDisor",
    "MDD",
    "PTSD",
    "LifetimeAntipsych",
    "LifetimeAnticonvulsant",
    "LifetimeAntidepress",
    "LifetimeLithium",
    "Tobbaco_ATOD",
    "Alcohol",
    "Opioid",
    "Opioid_Tox_bi",
    "Sedative_hypnotics",
    "Cannabis",
    "Amphetamine",
    "Cocaine",
    "Hallucinogens",
    "Inhalant",
    "Polysubstance"
]

# Columns to convert
cols_to_convert = phenotype_cols + ['Sex']

for col in cols_to_convert:
    if col in binary_df2.columns:
        # If bool, convert directly to int
        if binary_df2[col].dtype == 'bool':
            binary_df2[col] = binary_df[col].astype(int)
        # If object/string, convert 'Yes'/'No' and also 'Male'/'Female' for Gender
        elif binary_df2[col].dtype == 'object':
            if col == 'Sex':
                binary_df2[col] = binary_df[col].map({'Male': 1, 'Female': 0})
            else:
                binary_df2[col] = binary_df[col].map({'Yes': 1, 'No': 0})
        else:
            # If already numeric, no change
            pass

# Quick check of unique values after conversion
for col in cols_to_convert:
    if col in binary_df2.columns:
        print(f"{col} unique values: {binary_df2[col].unique()}")

# Add the Age2 variable
binary_df2['Age2'] = binary_df2['AgeDeath']**2
suffixes = ["_9", "_24", "_25", "_11"]
def extract_region(sample_id):
    for suf in suffixes:
        if sample_id.endswith(suf):
            return suf.replace("_", "BA")  # e.g., _11 → BA11
    return "Other"
# Extract BrainRegion
binary_df2["BrainRegion"] = binary_df2["SampleID"].astype(str).apply(extract_region)
# Make dummies fresh
binary_df2 = pd.get_dummies(binary_df2, columns=["BrainRegion"])

# Transform boolean
for col in region_covars:
    binary_df2[col] = binary_df2[col].astype(int)

# Update covariates list to include brain region dummies
region_covars = [c for c in binary_df2.columns if c.startswith("BrainRegion_")]
phenotype_cols = [c.strip() for c in phenotype_cols]
phenotype_cols2 = phenotype_cols
# Convert Gender (and any other categorical covariates) to numeric BEFORE correlation
covariates = ['AgeDeath', 'Age2', 'Sex', 'RIN', 'PMI', 'ast', 'end', 'mic', 'neu', 'oli', 'opc',' W_1', 'W_2'] + region_covars
covariates2 = [c.strip() for c in covariates]

COD_drugs_bi unique values: [ 0.  1. nan]
COD_Suicide_bi unique values: [ 0.  1. nan]
PsychiatriDisor unique values: [0 1]
MDD unique values: [0 1]
PTSD unique values: [0 1]
LifetimeAntipsych unique values: [ 0. nan  1.]
LifetimeAnticonvulsant unique values: [ 0. nan  1.]
LifetimeAntidepress unique values: [ 0.  1. nan]
LifetimeLithium unique values: [ 0. nan  1.]
Tobbaco_ATOD unique values: [ 0.  1. nan]
Alcohol unique values: [0 1]
Opioid unique values: [0 1]
Opioid_Tox_bi unique values: [ 0.  1. nan]
Sedative_hypnotics unique values: [0 1]
Cannabis unique values: [0 1]
Amphetamine unique values: [0 1]
Cocaine unique values: [0 1]
Hallucinogens unique values: [0 1]
Inhalant unique values: [0]
Polysubstance unique values: [0 1]
Sex unique values: [1 0]


In [123]:
brain_clocks = ['KPANN_brain_clock',
 'DESeq2_brain_clock',
 'Pearson_brain_clock',
 'Dev_brain_clock',
 'deMagalhaes_brain_clock',
 'GenAge_brain_clock',
 'GTExAge_brain_clock',
 'Peters_brain_clock',
 'all_brain_clock']
brain_clocks

['KPANN_brain_clock',
 'DESeq2_brain_clock',
 'Pearson_brain_clock',
 'Dev_brain_clock',
 'deMagalhaes_brain_clock',
 'GenAge_brain_clock',
 'GTExAge_brain_clock',
 'Peters_brain_clock',
 'all_brain_clock']

In [124]:
# -------------------------------
# Run forward stepwise for each brain clock
# -------------------------------
results_step_age2_region = []
for clock in brain_clocks:
    try:
        selected = forward_stepwise_selection(
            data=binary_df2,
            response=clock,
            predictors=phenotype_cols2,
            covariates=covariates2,
            verbose=False
        )

        if not selected:  # safety check
            print(f"No predictors selected for {clock}")
            continue

        formula = f"{clock} ~ {' + '.join(selected)}"
        model = smf.ols(formula=formula, data=binary_df2).fit()

        for var in selected:
            if var == "Intercept":
                continue
            try:
                coef = model.params[var]
                pval = model.pvalues[var]
                conf_int = model.conf_int().loc[var]
                tval = model.tvalues[var]

                results_step_age2_region.append({
                    'Clock': clock,
                    'Variable': var,
                    'Beta': coef,
                    'SE': model.bse[var],
                    't-value': tval,
                    'P-value': pval,
                    'CI_lower_95': conf_int[0],
                    'CI_upper_95': conf_int[1],
                    'R2': model.rsquared,
                    'Adj_R2': model.rsquared_adj,
                    'N': int(model.nobs),
                    'Selected_Model': formula
                })
            except KeyError:
                print(f"⚠️ Variable {var} not found in model for {clock}")

    except Exception as e:
        print(f"Stepwise failed for {clock}: {e}")
        results_step_age2_region.append({
            'Clock': clock,
            'Variable': None,
            'Beta': np.nan,
            'SE': np.nan,
            't-value': np.nan,
            'P-value': np.nan,
            'CI_lower_95': np.nan,
            'CI_upper_95': np.nan,
            'R2': np.nan,
            'Adj_R2': np.nan,
            'N': np.nan,
            'Selected_Model': None
        })

# -------------------------------
# Save results
# -------------------------------
results_df_step_age2_region = pd.DataFrame(results_step_age2_region)

In [125]:
# Save results with tissue information
results_df_step_age2_region.to_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/03vabb_assoc_forward_age2_region-09012025.csv", index=False)
results_df_step_age2_region

,Clock,Variable,Beta,SE,t-value,P-value,CI_lower_95,CI_upper_95,R2,Adj_R2,N,Selected_Model
0,KPANN_brain_clock,AgeDeath,0.249282,0.043591,5.718683,1.841557e-08,0.163640,0.334924,0.099604,0.069234,522,KPANN_brain_clock ~ AgeDeath + Age2 + Sex + RI...
1,KPANN_brain_clock,Age2,-0.024407,0.031892,-0.765301,4.444507e-01,-0.087065,0.038251,0.099604,0.069234,522,KPANN_brain_clock ~ AgeDeath + Age2 + Sex + RI...
2,KPANN_brain_clock,Sex,0.034736,0.092484,0.375586,7.073825e-01,-0.146966,0.216438,0.099604,0.069234,522,KPANN_brain_clock ~ AgeDeath + Age2 + Sex + RI...
3,KPANN_brain_clock,RIN,-0.002256,0.044792,-0.050359,9.598561e-01,-0.090258,0.085746,0.099604,0.069234,522,KPANN_brain_clock ~ AgeDeath + Age2 + Sex + RI...
4,KPANN_brain_clock,PMI,-0.033013,0.043734,-0.754859,4.506861e-01,-0.118938,0.052911,0.099604,0.069234,522,KPANN_brain_clock ~ AgeDeath + Age2 + Sex + RI...
...,...,...,...,...,...,...,...,...,...,...,...,...
188,all_brain_clock,Cannabis,0.189788,0.258413,0.734439,4.633697e-01,-0.319155,0.698732,0.736079,0.712854,273,all_brain_clock ~ AgeDeath + Age2 + Sex + RIN ...
189,all_brain_clock,LifetimeAnticonvulsant,-0.364555,0.093856,-3.884177,1.315112e-04,-0.549404,-0.179705,0.736079,0.712854,273,all_brain_clock ~ AgeDeath + Age2 + Sex + RIN ...
190,all_brain_clock,Alcohol,-0.351797,0.099660,-3.529987,4.945738e-04,-0.548077,-0.155518,0.736079,0.712854,273,all_brain_clock ~ AgeDeath + Age2 + Sex + RIN ...
191,all_brain_clock,LifetimeAntipsych,0.364228,0.108915,3.344131,9.520762e-04,0.149719,0.578737,0.736079,0.712854,273,all_brain_clock ~ AgeDeath + Age2 + Sex + RIN ...
